In [2]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Folder → AutoFix with Hard Pre-Gain + Lookahead Limiter → MP3 320) -----######-----###### #
import os, re, math, subprocess, tempfile
from pathlib import Path
import numpy as np
import soundfile as sf
from tqdm import tqdm

# -------------------- helpers -------------------- #
def _parse_leading_int(name):
    m = re.match(r"\s*(\d+)", Path(name).stem)
    return int(m.group(1)) if m else None

def _db_to_lin(db): 
    return 10.0**(db/20.0)

def _peak_dbfs(x):
    pk = float(np.max(np.abs(x)) + 1e-12)
    return 20.0 * math.log10(pk)

def _estimate_lufs_like(x, sr):
    # quick/stable proxy for loudness (not BS.1770)
    fc = 100.0
    alpha = math.exp(-2.0*math.pi*fc/sr)
    src = np.abs(x) if x.ndim==1 else np.mean(np.abs(x), axis=1)
    y = np.empty_like(src, dtype=float)
    prev = 0.0
    for i, s in enumerate(src):
        prev = (1-alpha)*s + alpha*prev
        y[i] = prev
    rms = float(np.sqrt(np.mean(y**2)))
    return float(-0.691 + 20.0*math.log10(rms + 1e-12))

def _hp_filter_inplace(x, sr, fc=20.0):
    # 1st order HP (IIR) to remove DC/rumble; per-channel
    if x.ndim == 1:
        x = x[:, None]
    a = math.exp(-2.0*math.pi*fc/sr)
    y = np.empty_like(x)
    for c in range(x.shape[1]):
        prev = 0.0
        for i in range(x.shape[0]):
            prev = (1-a)*x[i, c] + a*prev
            y[i, c] = x[i, c] - prev
    return y.squeeze()

def _detect_clipped_runs(x, thr=0.985):
    a = np.abs(x) >= thr
    runs = []; in_run = False; s = 0
    for i, v in enumerate(a):
        if v and not in_run:
            in_run = True; s = i
        elif not v and in_run:
            in_run = False; runs.append((s, i-1))
    if in_run:
        runs.append((s, len(a)-1))
    return runs

def _cubic_interp_segment(x, s, e):
    n = len(x); L, R = s-1, e+1
    if L < 0 or R >= n:
        span = e - s + 1
        fill = x[R] if s == 0 and R < n else (x[L] if e == n-1 and L >= 0 else 0.0)
        x[s:e+1] = fill; 
        return
    y0, y1 = x[L], x[R]
    span = e - s + 1
    t = np.linspace(0, 1, span)
    ssm = t*t*(3 - 2*t)
    x[s:e+1] = (1-ssm)*y0 + ssm*y1

def _light_declip(x, declip_thresh=0.985):
    y = x.copy()
    runs = _detect_clipped_runs(y, thr=declip_thresh)
    for (s, e) in runs:
        _cubic_interp_segment(y, s, e)
    clip_density = sum((e - s + 1) for s, e in runs) / max(1, y.size)
    return y, float(clip_density)

def _apply_pregain(x, pre_db):
    return x * _db_to_lin(pre_db)

def _crest_factor_db(x):
    # crest = peak / rms in dB
    peak = np.max(np.abs(x)) + 1e-12
    rms = np.sqrt(np.mean(x**2)) + 1e-12
    return 20.0 * math.log10(peak / rms)

def _lookahead_limiter(x, sr, ceiling_dbfs=-2.0, lookahead_ms=5.0, release_ms=50.0):
    """
    Simple peak limiter with lookahead. Works per-channel.
    """
    if x.ndim == 1:
        x = x[:, None]
    N = x.shape[0]; C = x.shape[1]
    ceiling = _db_to_lin(ceiling_dbfs)
    la = max(1, int(sr * lookahead_ms / 1000.0))
    rel_a = math.exp(-1.0 / max(1, int(sr * release_ms / 1000.0)))

    y = np.empty_like(x)
    for c in range(C):
        ch = x[:, c]
        # moving max (abs) with lookahead using a simple rolling window
        absx = np.abs(ch)
        # Pad right for lookahead
        pad = np.pad(absx, (0, la), mode='edge')
        # Compute max over window [i, i+la)
        # Efficient enough with small la; otherwise could use deque
        env = np.maximum.accumulate(pad)
        env = np.maximum(env[la:] , absx)  # ensure at least current sample
        gain = np.ones(N, dtype=ch.dtype)
        g = 1.0
        for i in range(N):
            if env[i] > ceiling:
                target = ceiling / (env[i] + 1e-12)
                g = min(g, target)  # attack is instant
            else:
                g = 1.0 - (1.0 - g) * rel_a  # release
            gain[i] = g
        y[:, c] = ch * gain
    return y.squeeze()

def _peak_normalize(x, target_dbfs=-1.0):
    peak_now = float(np.max(np.abs(x)) + 1e-12)
    gain = _db_to_lin(target_dbfs) / peak_now
    return x * gain, float(20.0 * math.log10(gain + 1e-12))

def _ensure_stereo(x):
    return np.stack([x, x], axis=1) if x.ndim == 1 else x

def _write_wav(path, x, sr):
    sf.write(str(path), np.clip(x, -1.0, 1.0), sr, subtype='PCM_16')

def _ffmpeg_export_mp3(wav_path, mp3_path, bitrate_kbps=320):
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(wav_path),
        "-vn", "-c:a", "libmp3lame", "-b:a", f"{int(bitrate_kbps)}k",
        str(mp3_path)
    ], check=True)

def _decode_to_wav_temp(src_path, tmp_dir):
    """Decode MP3 → WAV using ffmpeg so we can read with soundfile safely."""
    dst = Path(tmp_dir) / (Path(src_path).stem + "_dec.wav")
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(src_path), "-vn", "-acodec", "pcm_s16le",
        str(dst)
    ], check=True)
    return dst

def _find_audio_files(folder):
    exts = (".wav", ".WAV", ".mp3", ".MP3")
    return [f for f in Path(folder).iterdir() if f.is_file() and f.suffix in exts]

def _choose_files(files):
    wavs = [f for f in files if f.suffix.lower() == ".wav"]
    if len(wavs) >= 2:
        with_int = [(f, _parse_leading_int(f.name)) for f in wavs]
        one = [f for f, n in with_int if n == 1]
        two = [f for f, n in with_int if n == 2]
        if one and two:
            return [one[0], two[0]], "concat"
        order = sorted(wavs, key=lambda f: (_parse_leading_int(f.name) is None,
                                            _parse_leading_int(f.name) or 999999,
                                            f.name))
        return order[:2], "concat"
    # otherwise single
    return [sorted(files, key=lambda f: f.name)[0]], "single"

# -------------------- MAIN -------------------- #
# -----######-----###### CORE IMPORTABLE FUNCTION -----######-----###### #
def _mixfix_0209_folder_GET_mp3320_autofix_strong(
    in_folder, 
    out_mp3_path,
    declip_thresh=0.985,
    limiter_ceiling_dbfs=-2.0,
    peak_target_dbfs=-1.0,
    mp3_bitrate_kbps=320
):
    """
    Folder → (1 file or 2+ WAV concat) → Aggressive pre-gain → De-clip → HP/DC → Lookahead limiter → Peak normalize → MP3 320
    Returns a log dict with detailed stats.
    """
    files = _find_audio_files(in_folder)
    if not files:
        raise FileNotFoundError(f"No WAV/MP3 files found in {in_folder}")

    with tempfile.TemporaryDirectory() as td:
        # Decide inputs
        chosen, mode = _choose_files(files)
        decoded_paths = []
        for p in chosen:
            if p.suffix.lower() == ".mp3":
                decoded_paths.append(_decode_to_wav_temp(p, td))
            else:
                decoded_paths.append(p)

        # TQM
        pbar = tqdm(total=7 if mode == "concat" else 6, desc="TQM | StrongAutoFix", unit="step")

        # Load & concat (preserve SR/ch; resample not needed for typical case; we’ll unify channels)
        xs = []
        sr_ref = None
        for fp in decoded_paths:
            x, sr = sf.read(fp, always_2d=True)
            if sr_ref is None:
                sr_ref = sr
            # unify to stereo for stability in DSP
            x = _ensure_stereo(x.squeeze().astype(np.float32))
            xs.append(x)
        xcat = np.concatenate(xs, axis=0) if len(xs) > 1 else xs[0]
        pbar.update(1)

        # Aggressive pre-gain pass (based on clip % & crest factor)
        # Measure clip density at 0.985 and crest factor
        clip_runs_L = _detect_clipped_runs(xcat[:, 0], declip_thresh)
        clip_runs_R = _detect_clipped_runs(xcat[:, 1], declip_thresh)
        clip_density = (sum((e - s + 1) for s, e in clip_runs_L) + 
                        sum((e - s + 1) for s, e in clip_runs_R)) / (2.0 * max(1, xcat.shape[0]))
        crest = _crest_factor_db(np.mean(xcat, axis=1))

        pre_db = 0.0
        if clip_density > 0.01: pre_db -= 3.0
        if clip_density > 0.03: pre_db -= 6.0
        if clip_density > 0.08: pre_db -= 9.0
        # also if crest < 9 dB (very squashed), push further
        if crest < 9.0: pre_db -= 3.0
        xpre = _apply_pregain(xcat, pre_db)
        pbar.update(1)

        # De-clip per channel
        xdc = xpre.copy()
        clip_stats = []
        for c in range(xdc.shape[1]):
            xc, dens = _light_declip(xdc[:, c], declip_thresh=declip_thresh)
            xdc[:, c] = xc
            clip_stats.append({"ch": c, "clip_density_after_pregain": round(float(dens), 6)})
        pbar.update(1)

        # DC + HP
        xclean = _hp_filter_inplace(xdc, sr_ref, fc=20.0)
        pbar.update(1)

        # Lookahead limiter to ceiling (true-peak-ish safety before final normalize)
        xlim = _lookahead_limiter(xclean, sr_ref, ceiling_dbfs=limiter_ceiling_dbfs, lookahead_ms=5.0, release_ms=50.0)
        pbar.update(1)

        # Final peak normalize to -1.0 dBFS
        peak_before = round(_peak_dbfs(xlim), 3)
        xnorm, gain_db = _peak_normalize(xlim, target_dbfs=peak_target_dbfs)
        peak_after = round(_peak_dbfs(xnorm), 3)
        pbar.update(1)

        # Export MP3
        out_mp3_path = Path(out_mp3_path)
        out_mp3_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_wav = Path(td) / "final_norm.wav"
        _write_wav(tmp_wav, xnorm, sr_ref)
        _ffmpeg_export_mp3(tmp_wav, out_mp3_path, bitrate_kbps=mp3_bitrate_kbps)
        pbar.update(1)
        pbar.close()

        # Loudness proxy
        lufs_est = round(_estimate_lufs_like(np.mean(xnorm, axis=1), sr_ref), 2)

    return {
        "mode": mode,
        "ordered_files": [str(p) for p in chosen],
        "sr_hz": int(sr_ref),
        "peak_before_db": peak_before,
        "applied_gain_db": round(gain_db, 3),
        "final_peak_db": peak_after,
        "pre_attenuation_db": round(pre_db, 2),
        "clip_density_initial": round(float(clip_density), 6),
        "crest_db_initial": round(float(crest), 2),
        "clip_stats_after_pregain_declipping": clip_stats,
        "limiter_ceiling_dbfs": limiter_ceiling_dbfs,
        "lufs_estimate": lufs_est,
        "out_mp3_path": str(out_mp3_path)
    }
# -----######-----###### END CORE FUNCTION -----######-----###### #


In [4]:
in_folder = "/Users/yerik/Downloads/fix"
out_mp3_path = "/Users/yerik/Downloads/fix/SPKR_sep_1_FIXED.mp3"


log = _mixfix_0209_folder_GET_mp3320_autofix_strong(
    in_folder=in_folder,
    out_mp3_path=out_mp3_path,
    declip_thresh=0.995,        # detect more clipping
    limiter_ceiling_dbfs=-7.0,  # safer ceiling
    peak_target_dbfs=-4.0,      # final normalize lower
    mp3_bitrate_kbps=320
)
print("=== SUMMARY ===")
for k, v in log.items():
    print(f"{k}: {v}")


TQM | StrongAutoFix: 7step [18:34:20, 9551.57s/step]                                                          


=== SUMMARY ===
mode: single
ordered_files: ['/Users/yerik/Downloads/fix/SPKR_sep_1.mp3']
sr_hz: 48000
peak_before_db: -15.702
applied_gain_db: 11.702
final_peak_db: -4.0
pre_attenuation_db: -21.0
clip_density_initial: 0.425921
crest_db_initial: 2.47
clip_stats_after_pregain_declipping: [{'ch': 0, 'clip_density_after_pregain': 0.0}, {'ch': 1, 'clip_density_after_pregain': 0.0}]
limiter_ceiling_dbfs: -7.0
lufs_estimate: -13.44
out_mp3_path: /Users/yerik/Downloads/fix/SPKR_sep_1_FIXED.mp3
